# MLflow Example: Tracking Two Model Training Runs

In [12]:
! pip3 install sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [13]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [14]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [15]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [16]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [17]:
! pip3 install mlflow

In [18]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [19]:
!pip3 install dagshub

In [20]:
import dagshub
dagshub.init(repo_owner='wrrrq', repo_name='mlflow-demo', mlflow=True)

Initialized MLflow to track repo "wrrrq/mlflow-demo"

Repository wrrrq/mlflow-demo initialized!

In [21]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [38]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/b21f7636db4043dfb25bd3a1247a38a8', creation_time=1792118858558, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1792118858558, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

In [ ]:
# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(): # <--
        # Train a RandomForestClassifier
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(model, name=run_name, skops_trusted_types=["sklearn.tree._tree.Tree"])

        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")


In [41]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

Training and Logging Model 1...


2026/10/16 11:14:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389
🏃 View run gentle-fish-355 at: https://dagshub.com/wrrrq/mlflow-demo.mlflow/#/experiments/0/runs/cac68a9a6b3b4fc7940494d71fee5f8e
🧪 View experiment at: https://dagshub.com/wrrrq/mlflow-demo.mlflow/#/experiments/0


In [43]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")


Training and Logging Model 2...


2026/10/16 11:18:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run able-steed-271 at: https://dagshub.com/wrrrq/mlflow-demo.mlflow/#/experiments/0/runs/94ddb601918040e8a4200449abd41de2
🧪 View experiment at: https://dagshub.com/wrrrq/mlflow-demo.mlflow/#/experiments/0


RestException: INVALID_PARAMETER_VALUE: Response: {'error_code': 'INVALID_PARAMETER_VALUE'}

In [26]:
# Instructions to visualize results
print("\nTo view the results, run the following command in your terminal:")
print("mlflow ui")
print("Then navigate to http://127.0.0.1:5000 to explore the experiment results.")


To view the results, run the following command in your terminal:
mlflow ui
Then navigate to http://127.0.0.1:5000 to explore the experiment results.
